<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/book_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Book Cipher / Running Key Cipher

## History
Book Ciphers have been used for centuries, long before modern cryptography existed. Spies and soldiers used them because the key material was something completely ordinary, a book that both sides already owned. No suspicious looking secret key ever needed to be carried around, since the key was hiding in plain sight on a bookshelf. Benedict Arnold famously used a book cipher during the American Revolutionary War, using a specific edition of a dictionary as his key source.

## What is a Book Cipher / Running Key Cipher?
These two names are often used for the same idea, and that idea is: **use a long piece of real text as the key**, instead of a short repeating keyword like in Vigenere Cipher.

*   In a **Book Cipher**, the two people agree on a specific book (and often a specific edition, since page numbers must match exactly). Some versions of a book cipher replace each plaintext word with a page-line-word reference instead of doing character math at all.
*   In a **Running Key Cipher**, which is what this notebook implements, the text of the book is used exactly like the Vigenere key, character by character, added using modular arithmetic. The important difference from Vigenere is that the key is **not repeated**. It just keeps running forward through the book, one character at a time, for as long as the message needs.

This is a big improvement over Vigenere Cipher. Since the key never repeats in a short cycle, the usual Index of Coincidence attack that breaks Vigenere (by finding a repeating key length) does not work here in the same simple way.

In this implementation, we use the printable ASCII range from space (` `) to tilde (`~`), which spans from ASCII value 32 to 126 ($N=95$). Both the plaintext and the book text used as the key must be made up of characters from this same range.

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $N = E - S + 1 = 95$ (Total number of printable characters)
*   $B$ = the book text (the key source), a long piece of ordinary text
*   $o$ = the starting offset, which character of the book the key stream begins from
*   $x$ = Numeric value of the plaintext character ($0 \le x < N$)
*   $y$ = Numeric value of the ciphertext character ($0 \le y < N$)
*   $k_i$ = Numeric value of the book character used at position $i$

### 1. Building the Key Stream
Starting at offset $o$ inside the book text, the key stream is simply the book's own characters, read forward one at a time:

$$k_i = B[\, (o + i) \bmod \text{len}(B) \,]$$

The modulo is only there so the code does not crash if the book text happens to be shorter than the plaintext. In real world use, the book should always be **much longer** than the message, so the key stream should never actually need to wrap back to the beginning.

### 2. Encryption
Exactly the same formula as Vigenere Cipher, except the key stream comes from a book instead of a repeating keyword:

$$E(x_i) = (x_i + k_i) \pmod N$$

To get the final ASCII value: $C = E(x_i) + S$

### 3. Decryption
$$D(y_i) = (y_i - k_i) \pmod N$$

To get the final ASCII value: $P = D(y_i) + S$

### Key Requirements
*   Both sides must agree on the **exact same book text**, character for character, including punctuation and spacing.
*   Both sides must agree on the **starting offset**, which character of the book to begin from.
*   The book text should be **long enough** to cover the whole message without wrapping around, otherwise part of the key repeats, and that reused part becomes weak again, just like Vigenere.
*   Unlike One-Time Pad, the key here is **not truly random**, it is normal human language. That means the key itself has letter frequency patterns, which is a weakness a determined attacker can sometimes use, especially if they can guess which book was used.

### 1. Import Dependencies

In [1]:
import random

### 2. Helper Utilities

In [2]:
START_ASCII = 32
END_ASCII = 126
TOTAL_CHAR = END_ASCII - START_ASCII + 1  # 95 characters

def validate_printable_text(text: str) -> None:
    for ch in text:
        code = ord(ch)
        if not (START_ASCII <= code <= END_ASCII):
            raise ValueError(
                f"Character {ch!r} (ASCII {code}) is outside the supported range "
                f"{START_ASCII}-{END_ASCII}."
            )

def get_key_stream(book_text: str, start_offset: int, length: int) -> str:
    # walks forward through the book text starting at start_offset
    # wraps around only if the book is shorter than the message, which should be avoided in real use
    validate_printable_text(book_text)
    if len(book_text) == 0:
        raise ValueError("'book_text' cannot be empty.")

    key_stream = ""
    index = start_offset
    while len(key_stream) < length:
        key_stream += book_text[index % len(book_text)]
        index += 1

    return key_stream

### 3. Our Sample "Book" (Key Source)

This is a short public domain passage from Alice's Adventures in Wonderland by Lewis Carroll, used here as a stand-in for a real book. In real use, both people would agree on a full length book, and probably a much longer starting offset.

In [3]:
BOOK_TEXT = (
    "Alice was beginning to get very tired of sitting by her sister on the bank, "
    "and of having nothing to do. Once or twice she had peeped into the book her "
    "sister was reading, but it had no pictures or conversations in it, and what "
    "is the use of a book, thought Alice, without pictures or conversations. So "
    "she was considering in her own mind, as well as she could, for the hot day "
    "made her feel very sleepy and stupid, whether the pleasure of making a "
    "daisy chain would be worth the trouble of getting up and picking the "
    "daisies, when suddenly a White Rabbit with pink eyes ran close by her."
)

print(f"Book length: {len(BOOK_TEXT)} characters")

Book length: 588 characters


### 4. Generate a Random Key (Starting Offset)

In [4]:
def generate_random_key(book_text: str, message_length: int) -> int:
    # picks a random starting point in the book, leaving enough room to avoid wrapping if possible
    if message_length >= len(book_text):
        return random.randint(0, len(book_text) - 1)
    return random.randint(0, len(book_text) - message_length)

### 5. Encryption

In [5]:
def encrypt(text: str, book_text: str, start_offset: int = 0) -> str:
    validate_printable_text(text)
    key_stream = get_key_stream(book_text, start_offset, len(text))

    encrypted_text = ""
    for i, ch in enumerate(text):
        code = ord(ch)
        shift = ord(key_stream[i]) - START_ASCII
        cipher_code = ((code - START_ASCII) + shift) % TOTAL_CHAR
        encrypted_text += chr(cipher_code + START_ASCII)

    return encrypted_text

### 6. Decryption

In [6]:
def decrypt(cipher_text: str, book_text: str, start_offset: int = 0) -> str:
    validate_printable_text(cipher_text)
    key_stream = get_key_stream(book_text, start_offset, len(cipher_text))

    decrypted_text = ""
    for i, ch in enumerate(cipher_text):
        code = ord(ch)
        shift = ord(key_stream[i]) - START_ASCII
        plain_code = ((code - START_ASCII) - shift) % TOTAL_CHAR
        decrypted_text += chr(plain_code + START_ASCII)

    return decrypted_text

### 7. Example usage

In [7]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""

key = generate_random_key(BOOK_TEXT, len(plaintext))
print(f"Generated Random Key (starting offset into the book): {key}")
print(f"Key Stream Used: {get_key_stream(BOOK_TEXT, key, len(plaintext))!r}")

Generated Random Key (starting offset into the book): 490
Key Stream Used: ' getting up and picking the daisies, when suddenly a White Rabbit wit'


In [8]:
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, BOOK_TEXT, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, BOOK_TEXT, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: T76thORZejpMCbXaXOdk+VMnihv0umijSY]! 9[KP )'dlx&Q+4h0Y7i&v57uzi|%"Or#
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 8. What Happens If Both Sides Do Not Agree on the Same Book

This cell shows why the exact book and offset matter so much. Decrypting with even a slightly different starting offset produces complete garbage, not something close to the original.

In [9]:
wrong_offset = key + 1  # off by just one character
wrong_decryption = decrypt(cipher_text, BOOK_TEXT, wrong_offset)

print(f"Correct offset ({key}):    {decrypted_text}")
print(f"Wrong offset ({wrong_offset}):      {wrong_decryption}")

Correct offset (490):    TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Wrong offset (491):      lQA ~`jZoypkT}Xpnkx"<nMy!#vK4$u!meQ!(Pu\P,3B '*9W+RhXpMt@vbU38 (%*e}:
